# 05 - Cleaning Eurostat Long-Term Residents for Cameroonian Citizens

This notebook cleans Eurostat data on long-term resident permits granted to Cameroonian citizens.

Analytical role:

- supports Q3 on post-arrival trajectories
- captures one form of longer-term settlement after arrival
- standardizes the dataset for integration into `post_arrival_master.csv`

The indicator is interpreted separately from first permits, status changes and citizenship acquisition because each measure describes a different administrative event.


## 1. Objective of the Notebook

The objective of this notebook is to clean and prepare Eurostat data on long-term residents for Cameroonian citizens.

This source helps identify how many Cameroonian citizens hold long-term resident status in European destination countries across available years.

The cleaned output produced in this notebook will later contribute to the construction of the `post_arrival_master.csv` analytical table.


## 2. Analytical Role in the Project

This dataset supports the third research question of the project:

**Q3: How did post-arrival trajectory indicators evolve across the available 2015-2024 data period?**

Long-term residence is used as an administrative indicator of more durable settlement after arrival.

It should be interpreted separately from first permits, status changes and citizenship acquisition because each indicator measures a different administrative event.


## 3. Source File Used

The raw file used in this notebook comes from Eurostat data on long-term residents.

Raw input file:

```text
data/raw/europe/eurostat_long_term_residents_cameroon.xlsx
```

Cleaned output file:

```text
data/processed/cleaned/eurostat_long_term_residents_cleaned.csv
```

The raw file is not modified directly. All transformations are performed in this notebook and exported as a cleaned CSV file.


## 4. Environment Setup and Paths

This section imports the required Python library and defines the project paths.

The notebook uses `pandas` for data manipulation and `pathlib` to manage file paths in a more robust and readable way.

The cleaned output folder is created automatically if it does not already exist.


In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

## 5. Data Loading

In this section, the Excel file is loaded and the available sheets are inspected.

This step is necessary because Eurostat Excel exports may contain metadata rows, empty columns, footnotes or formatting elements that are not directly usable for analysis.


In [2]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx")
xls.sheet_names

c:\Users\Linno\OneDrive\Documents\my_projects\cameroonian-migration-analysis\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3']

## 6. Initial Data Inspection

The raw sheet is loaded without assuming that the first row already contains clean column names.

The goal is to inspect the structure of the file before applying transformations, especially:

- metadata rows at the top of the sheet
- unnecessary columns
- year columns
- destination country labels
- long-term resident values
- empty or non-analytical rows


In [3]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_long_term_residents_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,Data extracted on 09/04/2026 14:04:03 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,Long-term residents by citizenship on 31 Decem...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,23/03/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Legal framework,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Country of citizenship,NaN,Cameroon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Unit of measure,NaN,Person,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,TIME,2015,NaN,2016,NaN,2017,NaN,2018,NaN,2019,...,2020,NaN,2021,NaN,2022,NaN,2023,NaN,2024,NaN


## 7. Cleaning Strategy

The cleaning strategy focuses on transforming the Eurostat export into a clean analytical dataset.

The main actions include:

- removing unnecessary metadata rows
- removing unused columns
- keeping relevant destination country rows
- replacing Eurostat missing-value symbols
- using the correct row as the column header
- renaming columns clearly
- reshaping the data from wide format to long format
- converting year and value fields into appropriate data types


In [4]:
sheet1_raw = sheet1_raw.drop(
    columns=list(range(2, 21, 2)),
    errors="ignore"
)

sheet1_raw = sheet1_raw.drop(
    index=list(range(0, 9)) + list(range(46, 52)),
    errors="ignore"
)

sheet1_raw = sheet1_raw.reset_index(drop=True)

sheet1_raw

,0,1,3,5,7,9,11,13,15,17,19
0,TIME,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,GEO (Labels),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,European Union - 27 countries (from 2020),40523,47511,49796,:,51601,52523,53111,59455,61734,63999
3,European Union - 28 countries (2013-2020),45370,52358,54643,:,:,:,:,:,:,:
4,Belgium,977,2943,2834,2802,2879,3058,3287,6719,7020,7570
5,Bulgaria,6,6,11,11,12,12,12,13,15,16
6,Czechia,53,56,58,59,60,62,70,73,79,80
7,Denmark,121,109,95,97,146,158,179,197,227,229
8,Germany,80,2456,2565,2722,2959,3068,2934,3533,4147,4277
9,Estonia,0,0,0,1,2,1,1,1,3,5


In [5]:
sheet1_raw = sheet1_raw.drop(index=[1, 2, 3, 37], errors="ignore").reset_index(drop=True)
sheet1_raw

,0,1,3,5,7,9,11,13,15,17,19
0,TIME,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,Belgium,977,2943,2834,2802,2879,3058,3287,6719,7020,7570
2,Bulgaria,6,6,11,11,12,12,12,13,15,16
3,Czechia,53,56,58,59,60,62,70,73,79,80
4,Denmark,121,109,95,97,146,158,179,197,227,229
5,Germany,80,2456,2565,2722,2959,3068,2934,3533,4147,4277
6,Estonia,0,0,0,1,2,1,1,1,3,5
7,Ireland,0,0,0,0,0,0,0,1,1,1
8,Greece,15,17,22,25,25,27,25,13,16,17
9,Spain,2176,4148,4314,4479,4553,4463,4286,4629,4678,4746


## 8. Handling Missing or Suppressed Values

Eurostat exports sometimes use symbols such as `:` to represent missing, unavailable or confidential values.

In this notebook, these values are replaced to make the dataset easier to process.

This decision should be interpreted carefully: a replaced value does not necessarily mean that the real value is zero. It means that the value was not available in the original export and was handled for consistency in the cleaned dataset.


In [6]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw

,0,1,3,5,7,9,11,13,15,17,19
0,TIME,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
1,Belgium,977,2943,2834,2802,2879,3058,3287,6719,7020,7570
2,Bulgaria,6,6,11,11,12,12,12,13,15,16
3,Czechia,53,56,58,59,60,62,70,73,79,80
4,Denmark,121,109,95,97,146,158,179,197,227,229
5,Germany,80,2456,2565,2722,2959,3068,2934,3533,4147,4277
6,Estonia,0,0,0,1,2,1,1,1,3,5
7,Ireland,0,0,0,0,0,0,0,1,1,1
8,Greece,15,17,22,25,25,27,25,13,16,17
9,Spain,2176,4148,4314,4479,4553,4463,4286,4629,4678,4746


## 9. Data Transformation

The dataset is transformed from wide format to long format.

The expected cleaned structure includes:

- `destination_country`: European destination country
- `year`: reference year
- `residents`: number of long-term residents
- `source`: data source
- `nature`: indicator type

This structure makes the dataset easier to combine later with other post-arrival indicators.


In [7]:
# Use the first row as the header
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

# Rename TIME to destination_country
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

# Reshape from wide format to long format
sheet1_raw = sheet1_raw.melt(
    id_vars=["destination_country"],
    var_name="year",
    value_name="residents"
)

# Clean data types
sheet1_raw["year"] = sheet1_raw["year"].astype(int)

sheet1_raw["residents"] = (
    pd.to_numeric(sheet1_raw["residents"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Reorder columns
sheet1_raw = sheet1_raw[
    ["destination_country", "year", "residents"]
]

sheet1_raw

,destination_country,year,residents
0,Belgium,2015,977
1,Bulgaria,2015,6
2,Czechia,2015,53
3,Denmark,2015,121
4,Germany,2015,80
...,...,...,...
325,Liechtenstein,2024,4
326,Norway,2024,49
327,Switzerland,2024,2528
328,United Kingdom,2024,0


In [8]:
sheet1_raw["source"] = "eurostat"
sheet1_raw["nature"] = "long_term_resident"

sheet1_raw


,destination_country,year,residents,source,nature
0,Belgium,2015,977,eurostat,long_term_resident
1,Bulgaria,2015,6,eurostat,long_term_resident
2,Czechia,2015,53,eurostat,long_term_resident
3,Denmark,2015,121,eurostat,long_term_resident
4,Germany,2015,80,eurostat,long_term_resident
...,...,...,...,...,...
325,Liechtenstein,2024,4,eurostat,long_term_resident
326,Norway,2024,49,eurostat,long_term_resident
327,Switzerland,2024,2528,eurostat,long_term_resident
328,United Kingdom,2024,0,eurostat,long_term_resident


In [9]:
sheet1_raw = sheet1_raw.rename(
    columns={"permit": "permits"}
)

sheet1_raw


,destination_country,year,residents,source,nature
0,Belgium,2015,977,eurostat,long_term_resident
1,Bulgaria,2015,6,eurostat,long_term_resident
2,Czechia,2015,53,eurostat,long_term_resident
3,Denmark,2015,121,eurostat,long_term_resident
4,Germany,2015,80,eurostat,long_term_resident
...,...,...,...,...,...
325,Liechtenstein,2024,4,eurostat,long_term_resident
326,Norway,2024,49,eurostat,long_term_resident
327,Switzerland,2024,2528,eurostat,long_term_resident
328,United Kingdom,2024,0,eurostat,long_term_resident


## 10. Quality Checks After Cleaning

After cleaning, the dataset should be checked to confirm that the output is reliable.

Recommended checks include:

- verifying the number of rows after filtering
- checking missing values in key columns
- checking duplicated records
- confirming that `year` is numeric
- confirming that `residents` is numeric
- verifying destination country coverage
- checking that the dataset only contains the intended indicator


## 11. Output Export

The cleaned dataset is exported to the processed data folder.

Output file:

```text
data/processed/cleaned/eurostat_long_term_residents_cleaned.csv
```

This cleaned file will later contribute to the construction of the `post_arrival_master.csv` analytical table.


In [10]:
sheet1_raw.to_csv(
    CLEANED_PATH / "eurostat_long_term_residents_cleaned.csv",
    index=False
)

## 12. Methodological Note: Long-Term Residence

Long-term residence is an administrative status.

It should not be interpreted as the full migration journey of Cameroonian migrants. It only captures one measurable post-arrival outcome available in Eurostat data.

A rise in long-term residents may indicate more durable settlement, but it does not directly explain the personal motivations of migrants or the exact date of their arrival.


## 13. Limitations

This dataset should be interpreted with caution.

Main limitations:

- it covers Eurostat countries only
- it measures administrative long-term resident status, not total migrant stock
- it does not measure first entry into a country
- it does not track individual migrant journeys
- missing or unavailable values may affect interpretation
- the data should not be used alone to infer direct Covid-19 causality


## 14. Conclusion

This notebook produces a cleaned version of the Eurostat long-term residents dataset for Cameroonian citizens.

The cleaned output is ready for later standardization and integration into the post-arrival analytical table.

This source contributes to the analysis of long-term settlement patterns among Cameroonian migrants in European destination countries.
